# 02 — Czyszczenie i preprocessing (notatnik roboczy)

Cel: przekształcić surowe pliki `Fake.csv` / `True.csv` w zwalidowany, oznaczony etykietami plik Parquet gotowy do fazy modelowania.

## Kontekst przejęty z wcześniejszych prac
- Wcześniejsza EDA oraz analiza leakage znajdują się w katalogach `notebooks/explore_temp/` i `notebooks/report_temp/` (opis zadania wskazywał na `notebooks/explore/01_eda.ipynb`, ale ten katalog był pusty — jedyne wcześniejsze notatniki znajdują się w katalogach `_temp`). Zgłaszam to jako otwartą kwestię.
- Istnieje już produkcyjny moduł czyszczący: `src/preprocessing/cleaning.py` z funkcjami `clean_text()` i `filter_short_articles()`. Koduje on wszystkie reguły usuwania leakage wyprowadzone z wcześniejszych analiz chi-kwadrat / MI / bigramów.
- Konwencja etykiet z wcześniejszego pipeline'u: **True=1, Fake=0**. Utrzymuję ją dla spójności z istniejącymi raportami.

## Znane problemy leakage / jakości (z wcześniejszej analizy — nie wyprowadzam ich ponownie)
- Znaczniki agencji prasowych: `reuters`, `(Reuters)`, `WASHINGTON (Reuters) -` — występują wyłącznie w klasie prawdziwej, chi2 ~22,7k.
- Artefakty URL: `https://`, `pic.twitter.com/`, `tmsnrt.rs/` — skośne w stronę fake (pic/https 3509x), skośne w stronę true (tmsnrt).
- Szablony podpisów zdjęć: `featured image via Getty`, `Photo by X/Getty`, `getty images`, `image/video`.
- Uchwyty społecznościowe: `@username` — skośne w stronę fake.
- Artefakty HTML/JS: `&amp;`, `<script>...`, `&quot;`.
- Specyficzne dla strony: `21st Century Wire`.
- Zniekształcone sklejenia data-słowo: `2017Trump` → `2017 Trump`.
- Zduplikowane wiersze między klasami i wewnątrz klas.
- Puste lub bardzo krótkie artykuły (szczególnie po usunięciu URL).

## Przepływ pracy w tym notatniku
1. Wczytanie obu CSV, profilowanie kształtu/typów/braków/duplikatów/długości tekstu.
2. Konkatenacja z etykietą, usunięcie duplikatów (również między klasami).
3. Zastosowanie `clean_text` z `src.preprocessing.cleaning`.
4. Odfiltrowanie artykułów krótszych niż 10 słów po czyszczeniu.
5. Walidacja: unikalne id, balans klas, brak pustych tekstów, kontrola resztkowego leakage, kontrakt schematu.
6. Zapis do pliku Parquet w `data/processed/news_cleaned.parquet`.

In [1]:
import sys
from pathlib import Path

# Make `src` importable regardless of how the notebook is launched.
PROJECT_ROOT = Path('/home/aoleszkiewicz/dev/factlens')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.preprocessing.cleaning import clean_text, filter_short_articles

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FAKE_PATH = RAW_DIR / 'Fake.csv'
TRUE_PATH = RAW_DIR / 'True.csv'
OUT_PATH = PROCESSED_DIR / 'news_cleaned.parquet'

pd.set_option('display.max_colwidth', 120)
print('pandas', pd.__version__)

pandas 2.3.3


## 1. Profilowanie surowych plików CSV

Zwięzłe profilowanie — liczba wierszy, typy, braki, duplikaty, rozkłady długości tekstu dla każdego pliku. Te liczby stanowią punkt odniesienia "przed", do którego porównujemy wyniki na końcu.

In [2]:
fake_raw = pd.read_csv(FAKE_PATH)
true_raw = pd.read_csv(TRUE_PATH)

for name, df in [('Fake', fake_raw), ('True', true_raw)]:
    print(f'--- {name} ---')
    print(f'  shape: {df.shape}')
    print(f'  dtypes:\n{df.dtypes.to_string()}')
    print(f'  nulls per col: {df.isna().sum().to_dict()}')
    print(f'  exact-row duplicates: {df.duplicated().sum()}')
    print(f'  duplicate text field: {df["text"].duplicated().sum()}')
    wc = df['text'].fillna('').str.split().str.len()
    print(f'  text word-count: min={wc.min()}, p50={int(wc.median())}, p95={int(wc.quantile(0.95))}, max={wc.max()}')
    print(f'  empty text rows: {(wc == 0).sum()}')
    print()

--- Fake ---
  shape: (23481, 4)
  dtypes:
title      object
text       object
subject    object
date       object
  nulls per col: {'title': 0, 'text': 0, 'subject': 0, 'date': 0}
  exact-row duplicates: 3
  duplicate text field: 6026


  text word-count: min=0, p50=363, p95=926, max=8135
  empty text rows: 630

--- True ---
  shape: (21417, 4)
  dtypes:
title      object
text       object
subject    object
date       object
  nulls per col: {'title': 0, 'text': 0, 'subject': 0, 'date': 0}
  exact-row duplicates: 206
  duplicate text field: 225


  text word-count: min=0, p50=359, p95=894, max=5172
  empty text rows: 1



In [3]:
# Peek at the 'subject' and 'date' columns — known to be class-leaking (subject values don't overlap between classes) and inconsistently formatted.
print('Fake subjects:', fake_raw['subject'].value_counts().to_dict())
print('True subjects:', true_raw['subject'].value_counts().to_dict())

# Quick look at date format divergence
print('\nFake date samples:', fake_raw['date'].head(3).tolist())
print('True date samples:', true_raw['date'].head(3).tolist())

Fake subjects: {'News': 9050, 'politics': 6841, 'left-news': 4459, 'Government News': 1570, 'US_News': 783, 'Middle-east': 778}
True subjects: {'politicsNews': 11272, 'worldnews': 10145}

Fake date samples: ['December 31, 2017', 'December 31, 2017', 'December 30, 2017']
True date samples: ['December 31, 2017 ', 'December 29, 2017 ', 'December 31, 2017 ']


### Decyzje wynikające z profilowania
- **Usuwamy kolumny `title`, `subject`, `date`** i zachowujemy jedynie `text` + `label`.
  - `subject`: słowniki kategorii nie mają części wspólnej między klasami Fake i True → 100% leakage klasowego, jeśli kolumnę pozostawimy.
  - `date`: formaty różnią się między klasami (kolejny kanał leakage), a same daty nie są kluczowe dla klasyfikacji opartej na treści.
  - `title`: również niesie stylistyczny leakage (wielkie litery, wzorce interpunkcji). Mógłby zostać ponownie rozważony jako osobna cecha; na razie trzymamy się wcześniejszego pipeline'u, który go odrzuca.
- Odpowiada to podejściu zakodowanemu już w `src/preprocessing/cleaning.py` oraz we wcześniejszych notatnikach raportowych.
- Otwarta kwestia na fazę raportową: czy chcemy zachować `title` jako opcjonalną kolumnę? Zaznaczone poniżej.

## 2. Oznaczanie etykietami, konkatenacja i deduplikacja

In [4]:
fake = fake_raw[['text']].copy()
fake['label'] = 0  # Fake = 0

true = true_raw[['text']].copy()
true['label'] = 1  # True = 1

raw_labeled = pd.concat([fake, true], ignore_index=True)
print('Before dedupe:', raw_labeled.shape)
print('Class balance before dedupe:')
print(raw_labeled['label'].value_counts().rename({0: 'fake', 1: 'true'}))

# Null check before deduplicating — a NaN text would collapse all NaNs into one row otherwise
print('\nNull text rows:', raw_labeled['text'].isna().sum())
raw_labeled = raw_labeled.dropna(subset=['text'])

Before dedupe: (44898, 2)
Class balance before dedupe:
label
fake    23481
true    21417
Name: count, dtype: int64

Null text rows: 0


In [5]:
# Dedupe on (text, label) first — exact-row dupes inside a class
dedup_within = raw_labeled.drop_duplicates(subset=['text', 'label'])
print('After within-class dedupe:', dedup_within.shape)
print('  dropped:', len(raw_labeled) - len(dedup_within))

# Check cross-class collisions: same text labeled both fake and true?
text_label_counts = dedup_within.groupby('text')['label'].nunique()
cross_class = text_label_counts[text_label_counts > 1]
print(f'\nTexts appearing with BOTH labels: {len(cross_class)}')
if len(cross_class) > 0:
    print('  -> these are ambiguous; dropping entirely to avoid polluting train/test splits.')
    ambiguous_texts = set(cross_class.index)
    deduped = dedup_within[~dedup_within['text'].isin(ambiguous_texts)].copy()
else:
    deduped = dedup_within.copy()

# Finally, drop any remaining text-only duplicates (should be zero after above logic)
deduped = deduped.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'\nFinal deduped shape: {deduped.shape}')
print(deduped['label'].value_counts().rename({0: 'fake', 1: 'true'}))

After within-class dedupe: (38647, 2)
  dropped: 6251

Texts appearing with BOTH labels: 1
  -> these are ambiguous; dropping entirely to avoid polluting train/test splits.

Final deduped shape: (38645, 2)
label
true    21191
fake    17454
Name: count, dtype: int64


**Uwaga o polityce deduplikacji.** Wcześniejszy pipeline w `report_temp/03_cleaning_pipeline.ipynb` wywołuje `.drop_duplicates()` na połączonej ramce *po* przypisaniu etykiet, co wychwytuje jedynie dokładne pary (text, label). Dodatkowo sprawdzam kolizje międzyklasowe, ponieważ takie wiersze są autentycznie niejednoznaczne i w cichy sposób obciążałyby dowolny model. Jeśli nie ma żadnych kolizji, zachowanie jest identyczne jak w poprzednim pipelinie.

## 3. Zastosowanie pipeline'u czyszczenia

Używamy `clean_text` z `src.preprocessing.cleaning` — pojedyncze źródło prawdy, bez przepisywania wyrażeń regularnych tutaj. Egzekwowane reguły (w kolejności):

1. Usunięcie znaczników agencji prasowych: `^CITY (Reuters) -`, `(Reuters|AP|AFP)`, samo `reuters`.
2. Usunięcie URL: `https?://`, `www.`, `pic.twitter.com/`, `tmsnrt.rs/`.
3. Usunięcie `@wzmianek`.
4. Usunięcie szablonów podpisów zdjęć z zachowaniem samodzielnych słów treściowych (`image`, `video`, `screenshot`).
5. Usunięcie `getty images`, zbitki `image/video`, `21st century wire`.
6. Usunięcie tagów HTML, bloków `<script>`, encji nazwanych (`&amp;` itp.).
7. Rozbicie zniekształconych sklejeń `YYYYWord`.
8. Zwinięcie białych znaków, usunięcie tokenów jednoznakowych, zamiana na małe litery.

In [6]:
cleaned = deduped.copy()
cleaned['text'] = cleaned['text'].astype(str).map(clean_text)

cleaned['word_count'] = cleaned['text'].str.split().str.len().fillna(0).astype(int)
print('Post-cleaning word count distribution:')
print(cleaned['word_count'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

Post-cleaning word count distribution:
count    38645.000000
mean       385.240988
std        302.260725
min          0.000000
1%          19.000000
5%          59.000000
50%        351.000000
95%        847.000000
99%       1261.560000
max       7770.000000
Name: word_count, dtype: float64


In [7]:
# Spot-check a couple of before/after rows
print('--- REAL example ---')
idx_real = cleaned[cleaned['label'] == 1].index[0]
print('BEFORE:', deduped.loc[idx_real, 'text'][:300])
print('AFTER :', cleaned.loc[idx_real, 'text'][:300])

print('\n--- FAKE example ---')
idx_fake = cleaned[cleaned['label'] == 0].index[0]
print('BEFORE:', deduped.loc[idx_fake, 'text'][:300])
print('AFTER :', cleaned.loc[idx_fake, 'text'][:300])

--- REAL example ---
BEFORE: WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 
AFTER : the head of  conservative republican faction in the .. congress, who voted this month for  huge expansion of the national debt to pay for tax cuts, called himself  “fiscal conservative” on sunday and urged budget restraint in 2018. in keeping with  sharp pivot under way among republicans, .. represe

--- FAKE example ---
BEFORE: Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger a
AFTER : donald trump just coul

## 4. Odfiltrowanie krótkich artykułów

Po czyszczeniu część wierszy jest niemal pusta (tylko URL, śmieci po scrapingu). Wcześniejsza analiza wybrała `< 10 słów` jako próg odrzucenia — zachowuję ten sam próg dla porównywalności z istniejącymi eksperymentami.

In [8]:
short = cleaned[cleaned['word_count'] < 10]
print(f'Short articles (< 10 words): {len(short)}')
print(short['label'].value_counts().rename({0: 'fake', 1: 'true'}).to_dict())
if len(short):
    print('\nSample short rows:')
    for _, r in short.head(5).iterrows():
        print(f"  [{'FAKE' if r['label']==0 else 'TRUE'}] ({r['word_count']}w) '{r['text'][:120]}'")

final = filter_short_articles(cleaned[['text', 'label']], min_words=10)
print(f'\nAfter short-article filter: {final.shape}')

# Post-cleaning dedupe: cleaning can collapse distinct raw texts into identical cleaned strings
# (e.g. two articles that differed only by Reuters dateline / URLs). Re-run the same
# cross-class-ambiguity check here before proceeding.
before = len(final)
tl_counts = final.groupby('text')['label'].nunique()
cross = tl_counts[tl_counts > 1]
print(f'Post-clean texts appearing with both labels: {len(cross)}')
if len(cross):
    final = final[~final['text'].isin(set(cross.index))]
final = final.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'Post-clean dedupe dropped: {before - len(final)} rows')
print(f'Shape after post-clean dedupe: {final.shape}')


Short articles (< 10 words): 170
{'fake': 170}

Sample short rows:
  [FAKE] (0w) ''
  [FAKE] (1w) 'enjoy:'
  [FAKE] (9w) 'watch my #openingstatment jeanine pirro () april , 2017'
  [FAKE] (0w) ''
  [FAKE] (0w) ''



After short-article filter: (38475, 2)
Post-clean texts appearing with both labels: 0
Post-clean dedupe dropped: 5 rows
Shape after post-clean dedupe: (38470, 2)


## 5. Kontrola resztkowego leakage

Szybka sonda dla tokenów o najwyższym ryzyku, które wskazała wcześniejsza analiza. Po czyszczeniu powinny wynosić ~0. Nie jest to pełne ponowne przeliczenie chi-kwadrat (to znajduje się w `report_temp/03_cleaning_pipeline.ipynb`), a jedynie zabezpieczenie kontrolne.

In [9]:
for token in ['reuters', 'getty', 'https', 'www.', 'pic.twitter', 'tmsnrt', '21st century wire', '@']:
    n = final['text'].str.contains(token, case=False, regex=False, na=False).sum()
    print(f"  '{token}': {n} articles contain it")

  'reuters': 16 articles contain it
  'getty': 30 articles contain it


  'https': 1 articles contain it
  'www.': 1 articles contain it


  'pic.twitter': 0 articles contain it
  'tmsnrt': 0 articles contain it
  '21st century wire': 0 articles contain it


  '@': 160 articles contain it


## 6. Nadanie stabilnych identyfikatorów i walidacja schematu

In [10]:
final = final.reset_index(drop=True)
final.insert(0, 'id', np.arange(len(final), dtype=np.int32))
final['label'] = final['label'].astype('int8')
final['text'] = final['text'].astype('string')  # pandas StringDtype -> parquet large_string

# Assertions — treat these as a data contract
assert final['id'].is_unique, 'id must be unique'
assert final['id'].notna().all(), 'id must be non-null'
assert final['label'].isin([0, 1]).all(), 'label must be binary {0,1}'
assert final['text'].notna().all(), 'text must be non-null'
assert (final['text'].str.split().str.len() >= 10).all(), 'all texts must have >= 10 words'
assert not final['text'].duplicated().any(), 'text must be unique after dedupe'

print('Schema locked:')
print(final.dtypes)
print('\nFinal shape:', final.shape)
print('Class balance:')
print(final['label'].value_counts().rename({0: 'fake', 1: 'true'}))

Schema locked:
id                int32
text     string[python]
label              int8
dtype: object

Final shape: (38470, 3)
Class balance:
label
true    21189
fake    17281
Name: count, dtype: int64


## 7. Uzgodnienie liczb względem źródła

In [11]:
raw_total = len(fake_raw) + len(true_raw)
print(f'Raw total rows:           {raw_total:,}')
print(f'After null drop + dedupe: {len(deduped):,}   (dropped {raw_total - len(deduped):,})')
print(f'After short-article filter: {len(final):,} (dropped {len(deduped) - len(final):,})')
print(f'\nFinal fake: {(final["label"]==0).sum():,}')
print(f'Final true: {(final["label"]==1).sum():,}')

Raw total rows:           44,898
After null drop + dedupe: 38,645   (dropped 6,253)
After short-article filter: 38,470 (dropped 175)

Final fake: 17,281
Final true: 21,189


## 8. Zapis do pliku Parquet

In [12]:
final.to_parquet(OUT_PATH, engine='pyarrow', compression='snappy', index=False)
print(f'Wrote {len(final):,} rows to {OUT_PATH}')
print(f'File size: {OUT_PATH.stat().st_size / 1e6:.2f} MB')

# Read back and verify schema round-trip
rt = pd.read_parquet(OUT_PATH)
print('\nRound-trip dtypes:')
print(rt.dtypes)
assert rt.shape == final.shape
assert (rt['id'] == final['id']).all()
print('\nRound-trip OK.')

Wrote 38,470 rows to /home/aoleszkiewicz/dev/factlens/data/processed/news_cleaned.parquet
File size: 53.58 MB



Round-trip dtypes:
id                int32
text     string[python]
label              int8
dtype: object

Round-trip OK.


## Otwarte kwestie na fazę raportową

1. **Brakujący notatnik, do którego było odniesienie.** Zadanie wskazywało `notebooks/explore/01_eda.ipynb`, jednak ten katalog był pusty — wcześniejsza EDA faktycznie znajduje się w `notebooks/explore_temp/01_eda.ipynb`. Czy katalogi `_temp` powinny zostać awansowane / scalone, czy `_temp` jest świadomym scratch-em?
2. **Usunąć `title`?** Wcześniejszy pipeline go odrzuca. Jest tam sygnał, ale również stylistyczny leakage (wzorce wielkich liter różnią się między klasami). Warte ponownego rozważenia jako osobna kolumna cechy przed modelowaniem.
3. **Próg krótkich artykułów.** 10 słów jest odziedziczone z wcześniejszej pracy bez analizy wrażliwości. Akceptowalne dla tej iteracji.
4. **Konwencja etykiet.** Utrzymuję True=1, Fake=0 dla spójności. Potwierdzić, że tego oczekuje kod modelujący.
5. **Ścieżka wyjściowa.** Wybrałem `data/processed/news_cleaned.parquet`. Istniejący CSV w `data/processed/cleaned_news_dataset.csv` został wyprodukowany przez starszy pipeline — zostaje na miejscu, nie jest nadpisywany.